In [ ]:
###Select dataset to analyze
path = 'Z:/Augusto/Images/NSPARC/20250731-NSPARC/mEGFP-KDM5A-mCherry' #avoid writing a dash / at the end

genotype = 'HEK293FT-mEGFP-KDM5A-mCherry'
dataset_number = 12

#Define which channel corresponds to fluorescent protein and which one corresponds to nucleus marker ('fp' or 'nucleus')
ch1 = 'nucleus'
ch2 = 'mEGFP'
ch3 = 'mCherry'

print("Analysis running")

Comments = 'N-terminus mEGFP and C-terminus mCherry-tagged KDM5A - NSPARC'

# import os
# print(os.listdir(path))

In [ ]:
# #####
#####Define parameters for cellpose
#####Cell detection

model_type = 'cyto' #nuclei', #'cyto', # cyto2 #models provided by cellpose
estimated_diameter = 275 #value 750 covers entire large cells SoRa; #275 covers small cells confocal; modify if necessary
flow_threshold = 0.4 #flow_threshold 0.4. Higher threshold enables more permissive cell detection
cellprob_threshold = 1 #1 works well 
contrast_stretching = 0 #takes percentiles on both edges of histogram (eg., for 5, 5th and 95th percentiles) values 0-100

In [ ]:
#####Run basic functions (readDataset, plot_image, implement_cellPose)
#####
#This function reads nd2 files generated by NSPARC in the Nachury lab.
#It takes the path to your files, the genotype (as my data is organized in folders named after the genotype of the imaged cells)
#and dataset number to select one of the datasets in the genotype folder

def readDataset(path, genotype_number):
    
    datafolder = os.path.join(path)
    #print(datafolder)
    os.chdir(datafolder)
    #print(os.getcwd())
    
    DatasetsinFolder = os.listdir()
    #print(DatasetsinFolder)
    
    for element in DatasetsinFolder:
        #print(element)
        if genotype_number in element:
            Dataset = element
            print("input file:", Dataset)
            break

    #Load the .nd2 file
    image_stack = pims.open(Dataset)
    return image_stack

#image_stack = readDataset(path, genotype_number)

#####
#####
#Daniel Foyt wrote this code to plot numpy arrays
def plot_image(image,title = '',size = 20,cmap='gray',cmapp_bar = True):
    fig, ax = plt.subplots(figsize=(size,size))
    plt.imshow(image, cmap=cmap)
    if cmapp_bar == True:
        cbar = plt.colorbar()
    ax.set_title(title)
    plt.show()
    
######
######
#set up cell_pose
#import cellpose
#from cellpose import models
##Cellpose was developed at janelia farm by Carsen Stringer and Marius Pachitariu
##https://doi.org/10.1101/2022.04.01.486764
#import torch
##t = torch.cuda.get_device_properties(0).total_memory
##r = torch.cuda.memory_reserved(0)
##a = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 8000000000 else "cpu")
#This code implements models for cell pose. This function allows to call cell pose later in the code. 
#It takes the model type used for the prediction of cell pose masks and an array (image) where the masks will be applied.
def implement_cellPose(model_type, estimated_diameter, flow_threshold, cellprob_threshold, image_slice):

    device = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 8000000000 else "cpu")
    device_type = device.type
    gpu = True if "cuda" in str(device_type) else False

    # https://github.com/MouseLand/cellpose/blob/main/cellpose/models.py
    model = models.Cellpose(
                model_type = model_type,
                device = device,
                gpu = gpu)

    estimated_diameter = estimated_diameter #estimated size of cells (pixels)

    # picked some image to test on 
    #sum_image_smoth = image[18][1]
    #sum_image_smoth2 = io.imread('D:/Augusto/Images/ConfocalImaging/20221115/mCherry-Nup98nt1833-KDM5Ant4666-spy650-640nm30_1/mCherry-Nup98nt1833-KDM5Ant4666-spy650-640nm30_1_MMStack_5-Pos000_000.ome.tif')[1][25] 

    sum_image_smoth = image_slice
    batch = [sum_image_smoth]

    #flow_threshold 0.75
    masks, flows, styles, diams = model.eval(batch,
                                                 diameter = estimated_diameter,
                                                 flow_threshold = flow_threshold,
                                                 cellprob_threshold = cellprob_threshold,
                                                 channels = None,
                                                 tile = False)
    
    return masks, flows, styles, diams, batch


In [ ]:
#####Select folder, dataset, and channels
#######
#######
#This code implements functions for reading tif.ome files
!pip install nd2reader
import nd2reader
import pims
import os
import re
import skimage 
from skimage import io
import matplotlib.pyplot as plt
import numpy as np
import tifffile as tf

#print(skimage.__version__)
#skimage reference
#Stéfan van der Walt, Johannes L. Schönberger, Juan Nunez-Iglesias, François Boulogne, Joshua D. Warner, Neil Yager, Emmanuelle Gouillart, Tony Yu and the scikit-image contributors. scikit-image: Image processing in Python. PeerJ 2:e453 (2014) https://doi.org/10.7717/peerj.453
#https://biomedicalhub.github.io/python-data/skimage.html - Read about image processing skimage

#path = 'Z:/Augusto/Images/NSPARC/20241011-NSPARC/mEGFP-KDM5A-mCherry' #avoid writing a dash / at the end
dataset_date = path.split('/')[-2]

dataset_number = str(dataset_number)
dataset_number = dataset_number.zfill(4)
genotype_number = genotype + '_' + dataset_number 

#Read the dataset
image_stack = readDataset(path, genotype_number)
print("\nfeatures:\n", image_stack, "\n") #length of stack is the number of slices

#Create annotation file
output_folder =  path + "/" + "analysis_" + dataset_date + '_' + genotype
filename_condensates = output_folder + "/" + dataset_date + '_' + genotype_number + '_condensates.txt'
filename_cells = output_folder + "/" + dataset_date + '_' + genotype_number + '_cells.txt'
filename_parameters = output_folder + "/" + dataset_date + '_' + genotype_number + '_parameters.txt'
print("output folder:", output_folder)
print("output files:", filename_condensates, "\n", filename_cells, "\n", filename_parameters, "\n")


#Assign proper channels to corresponding variables for further analysis

c_nucleus = 0
c_mEGFP = 1
c_mCherry = 2

# if ch1 == 'mEGFP' and ch2 == 'mCherry' and ch3 == 'nucleus':
#     c_fp = 0
#     c_nucleus = 1
# elif ch1 == 'nucleus' and ch2 == 'fp':
#     c_nucleus = 0
#     c_fp = 1
# else:
#     raise ValueError("ch values must be either 'fp' or 'nucleus'")


In [ ]:
####Choose slice to analyze
#Iterate over all cells

#image_stack length is the number of z-slice in the z stack
for z_slice in range(len(image_stack)):
    
    frame = image_stack.get_frame_2D(c=c_mEGFP, t=0, z=z_slice) #iterate t if needed
 
    #adjust contrast
    vmin = np.percentile(frame, 20)
    vmax = np.percentile(frame, 99.8)

    print("Slice ", z_slice)
    plt.figure(figsize=(10, 10))
    plt.imshow(frame, cmap='gray', vmin=vmin, vmax=vmax)
        
    plt.show()
    #Close the figure explicitly to free up memory
    plt.close()
    
    
define_slice = "Choose a slice for analysis\n"
analysis_slice = int(input(define_slice))

###Images (slices) for further analysis
image_chmEGFP_selected = np.array(image_stack.get_frame_2D(c=c_mEGFP, t=0, z=analysis_slice))
image_chmCherry_selected = np.array(image_stack.get_frame_2D(c=c_mCherry, t=0, z=analysis_slice))
image_chNuclei_selected = np.array(image_stack.get_frame_2D(c=c_nucleus, t=0, z=analysis_slice))

# ####
####Print the slice chosen for analysis

print("This is the slice that you chose\nslice: ", analysis_slice)

# Create a subplot with 1 row and 2 columns
fig, axes = plt.subplots(1, 3, figsize=(10, 5))

# Plot protein channel
vmin = np.percentile(image_chmEGFP_selected, 20)
vmax = np.percentile(image_chmEGFP_selected, 99.8)
axes[0].imshow(image_chmEGFP_selected, cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('mEGFP channel')

vmin = np.percentile(image_chmCherry_selected, 20)
vmax = np.percentile(image_chmCherry_selected, 99.8)
axes[1].imshow(image_chmCherry_selected, cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('mCherry channel')

# Plot chromatin channel
axes[2].imshow(image_chNuclei_selected, cmap='gray')
axes[2].set_title('Chromatin channel')

# Adjust layout for better spacing
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
#DEFINE SLICE FOR SPEED
###Images (slices) for further analysis
# image_chProtein_selected = image_chProtein[10]
# image_chNuclei_selected = image_chNuclei[10]

In [ ]:
#####Call cell pose on slices in a z-stack and select a slice
import cellpose
from cellpose import models
from cellpose import utils
from skimage import exposure
#Cellpose was developed at janelia farm by Carsen Stringer and Marius Pachitariu
#https://doi.org/10.1101/2022.04.01.486764

import torch
#t = torch.cuda.get_device_properties(0).total_memory
#r = torch.cuda.memory_reserved(0)
#a = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 8000000000 else "cpu")

print((np.shape(image_chNuclei_selected)))

#Contrast enhancing for spy650 channel
p_value_low = contrast_stretching
p_value_high = 100 - contrast_stretching
p_low, p_high = np.percentile(image_chNuclei_selected, (p_value_low, p_value_high))
#img_rescaled = exposure.rescale_intensity(image_chNuclei_selected, in_range=(p_low, p_high))
img_rescaled = image_chNuclei_selected

masks, flows, styles, diams, batch = implement_cellPose(model_type, estimated_diameter, flow_threshold, cellprob_threshold, img_rescaled)
#print(masks)

plot_image(batch[0],title = 'raw',size = 20,cmap='gray',cmapp_bar = False)
plot_image(masks[0]*100,title = 'masks',size = 20,cmap='cool',cmapp_bar = False)
outline_masks = cellpose.utils.masks_to_outlines(masks[0])
plot_image(outline_masks, title = 'masks',size = 20,cmap='cool',cmapp_bar = False)

print('Number of cells masked:', len(np.unique(masks[0]))-1) 
    # perimeter_masks = cellpose.utils.get_mask_perimeters(masks[0])
    # perimeter_masks
    
selected_slice_mask = masks[0]

# print('outline_masks', np.size(outline_masks))
# print(outline_masks)

    

In [ ]:
#####Match cellPose results and generate .txt output
#This chunk of code assings condensates to masks (cells)
#Initialize a condensates_stats np array to fill with various condensate values.
from skimage import measure
from skimage import morphology
from scipy.ndimage import center_of_mass
                           
#Plot fp and spy650 channels for reference
vmin = np.percentile(image_chmEGFP_selected, 20) #set contrast
vmax = np.percentile(image_chmEGFP_selected, 99.8) #set contrast
plt.imshow(image_chmEGFP_selected, vmin = vmin, vmax = vmax)
plt.title('mEGFP')
plt.show()
vmin = np.percentile(image_chmCherry_selected, 20) #set contrast
vmax = np.percentile(image_chmCherry_selected, 99.8) #set contrast
plt.imshow(image_chmCherry_selected, vmin = vmin, vmax = vmax)
plt.title('mCherry')
plt.show()
plt.imshow(image_chNuclei_selected)
plt.title('nucleus')
plt.show()

#print("condensate_stats", condensate_stats)
#find the unique masks in the array (number of masks) and iterate through them
unique_masks = np.unique(selected_slice_mask) 
print("unique masks:", unique_masks)

selected_cells_stats = np.empty((0, 8), float)
selected_coordinates = []
for mask_number in unique_masks:
    
    #####
    binary_cell_mask = selected_slice_mask == mask_number #make binary mask

    #Find mask coordinates
    mask_cell_rows, mask_cell_cols = np.where(binary_cell_mask == 1)
    y_low = np.min(mask_cell_rows) #Coordinates of area around mask
    y_up = np.max(mask_cell_rows)
    x_low = np.min(mask_cell_cols)
    x_up = np.max(mask_cell_cols)
    
    #Find center of mass for mask (center of cell)
    center_cell = center_of_mass(binary_cell_mask) #Calculate the center of mass of cell mask
    center_cell_y, center_cell_x = center_cell #Print the center coordinates
    center_cell_x = round(center_cell_x)
    center_cell_y = round(center_cell_y)
    
    #####
    mask_cell_rows, mask_cell_cols = np.where(binary_cell_mask == 1) #Find where mask is located
    
    if mask_number == 0:
        background_mEGFP = np.mean(image_chmEGFP_selected[mask_cell_rows, mask_cell_cols]) #1st mask is background 
        background_mCherry = np.mean(image_chmCherry_selected[mask_cell_rows, mask_cell_cols]) #1st mask is background 

    #Obtain stats for cell mask
    raw_mEGFP_cell_expression = np.sum(image_chmEGFP_selected[mask_cell_rows, mask_cell_cols])
    raw_mCherry_cell_expression = np.sum(image_chmCherry_selected[mask_cell_rows, mask_cell_cols])
    cell_area = np.sum(binary_cell_mask)
    
    ####Plot various stats of the cell mask currently analyzed
    
    #Create a figure with three subplots, for each of the segmented channels
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 10))
    contours_cell_mask = measure.find_contours(binary_cell_mask, 0.5)[0] #this line returns only the first contour detected
    vmin = np.percentile(image_chmEGFP_selected, 20) #set contrast
    vmax = np.percentile(image_chmEGFP_selected, 99.8) #set contrast
    ax1.imshow(image_chmEGFP_selected, origin='upper', vmin = vmin, vmax = vmax)
    ax1.plot(contours_cell_mask[:, 1], contours_cell_mask[:, 0], 'w', linewidth=1) #plot contours
    ax1.scatter(center_cell_x, center_cell_y, marker='*', color = "w", linestyle='None')
    ax1.set_xlim(x_low, x_up) #zoom to mask currently analyzed
    ax1.set_ylim(y_up, y_low)
    ax1.invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
    ax1.set_title("mask " + str(mask_number) + "\nfp")
    ax1.set_ylim(ax1.get_ylim()[::-1])  # Reverse the y-axis limits
    
    vmin = np.percentile(image_chmCherry_selected, 20) #set contrast
    vmax = np.percentile(image_chmCherry_selected, 99.8) #set contrast
    ax2.imshow(image_chmCherry_selected, origin='upper', vmin = vmin, vmax = vmax)
    ax2.plot(contours_cell_mask[:, 1], contours_cell_mask[:, 0], 'w', linewidth=1) #plot contours
    ax2.scatter(center_cell_x, center_cell_y, marker='*', color = "w", linestyle='None')
    ax2.set_xlim(x_low, x_up) #zoom to mask currently analyzed
    ax2.set_ylim(y_up, y_low)
    ax2.invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
    ax2.set_title("mask " + str(mask_number) + "\nfp")
    ax2.set_ylim(ax2.get_ylim()[::-1])  # Reverse the y-axis limits

    ax3.imshow(image_chNuclei_selected, origin='upper', cmap='viridis')
    ax3.plot(contours_cell_mask[:, 1], contours_cell_mask[:, 0], 'r', linewidth=1) #plot contours
    ax3.scatter(center_cell_x, center_cell_y, marker='*', color = "w", linestyle='None')
    ax3.set_xlim(x_low, x_up) #zoom to mask currently analyzed
    ax3.set_ylim(y_up, y_low)
    ax3.invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
    ax3.set_title("mask " + str(mask_number) + "\nspy650")
    ax3.set_yticklabels([])  # Remove y-axis tick labels
    ax3.set_ylim(ax3.get_ylim()[::-1])  # Reverse the y-axis limits
    
    plt.show()
    
    #plot different features of condensates associated with the mask currently analyzed
    print('mask id:', mask_number)
    mean_expression_mEGFP = raw_mEGFP_cell_expression / cell_area
    mean_expression_mCherry = raw_mCherry_cell_expression / cell_area

    print('mean mEGFP mask intensity (raw)', mean_expression_mEGFP)
    print('mean mCherry mask intensity (raw)', mean_expression_mCherry)
    
    print('background mEGFP:', background_mEGFP)
    print('background mCherry:', background_mCherry)
    
    if mask_number == 0:
        #Save nucleoplasm mask
        binary_no_cell_masks = binary_cell_mask
    elif mask_number > 0:
        #Ask if cell is good for further analysis
        ask_analyzed_cell = "Is this cell (mask) " + str(mask_number) + " good for further analysis (y/n)\n"
        answer_analyzed_cell = input(ask_analyzed_cell)
        
        if answer_analyzed_cell == 'y':
            
            selected_cells_stats = np.append(selected_cells_stats, np.array([[mask_number, 
                center_cell_x, center_cell_y, raw_mEGFP_cell_expression, raw_mCherry_cell_expression, 
                cell_area, background_mEGFP, background_mCherry]]), axis=0)
            
            #If selected, save coordinates for further plotting
            selected_coordinates.append([y_low,y_up,x_low,x_up])

print("\nAnalysis finished")

In [ ]:
###Save images of the cells selected
import matplotlib.colors
from matplotlib.patches import Rectangle
import io
import shutil

###This piece of code was taken from cle.imshow source code. It generates
###random colors for labels.
###Reference:
###https://github.com/clEsperanto/pyclesperanto_prototype/blob/571401a0ad215397423eda6833cd177e1086069b/pyclesperanto_prototype/_tier9/_imshow.py
from matplotlib.colors import ListedColormap
from numpy.random import MT19937
from numpy.random import RandomState, SeedSequence
rs = RandomState(MT19937(SeedSequence(3)))
lut = rs.rand(65537, 3)
lut[0, :] = 0
# these are the first four colours from matplotlib's default
lut[1] = [0.12156862745098039, 0.4666666666666667, 0.7058823529411765]
lut[2] = [1.0, 0.4980392156862745, 0.054901960784313725]
lut[3] = [0.17254901960784313, 0.6274509803921569, 0.17254901960784313]
lut[4] = [0.8392156862745098, 0.15294117647058825, 0.1568627450980392]
cmap  = matplotlib.colors.ListedColormap(lut)

####Create folders for analysis files
###
# Specify the folder path
if not os.path.exists(output_folder): # Check if the folder exists
    #Create the folder
    os.makedirs(output_folder)
    print("Folder created successfully!")
else:
    print("Folder already exists.")
    
#Create folder for all images
output_folder_images = output_folder + "/Images"

if not os.path.exists(output_folder_images): # Check if the folder exists
    os.makedirs(output_folder_images) #Create the folder
    print("Folder created successfully!")
else:
    print("Folder already exists.")

#If folder for images in dataset exists, delete and create new one
output_folder_images_dataset = output_folder + "/Images" + "/Images_" + str(dataset_date) + "-" + str(genotype_number)
print(output_folder_images_dataset)

if os.path.exists(output_folder_images_dataset): # Check if the folder exist
    shutil.rmtree(output_folder_images_dataset) #Delete the folder
    print("Folder already exists, folder deleted.")
    os.makedirs(output_folder_images_dataset) #Create the folder
    print("Folder created successfully!")
else:
    os.makedirs(output_folder_images_dataset)
    print("Folder created successfully!") #Create the folder

###Plot whole slice images
###
# Set the color for the contours overlay on nuclear image
contours_value = 64000 # white color
# Create a copy of the original image to overlay the contours
contours_image_chNuclei_selected = np.copy(image_chNuclei_selected) 
# Apply the contours mask to the overlay image
contours_image_chNuclei_selected[outline_masks] = contours_value

#Plot whole slice (nucleoplasm and cells with contours)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={'width_ratios': [1, 1]})
vmin_nuc, vmax_nuc = np.percentile(image_chNuclei_selected, (20, 99))

ax1.imshow(contours_image_chNuclei_selected, origin='upper', cmap = 'gray', vmin = vmin_nuc, vmax = vmax_nuc)
ax2.imshow(binary_no_cell_masks)

# Adjust spacing between subplots
plt.tight_layout()

# Generate name for whole slice image
titulo_slice = ("date: " +str(dataset_date) + "\ndataset: " + str(genotype_number) + "\nslice: " + str(analysis_slice) + 
                        "\nbackground_mEGFP: " + str(background_mEGFP) + "\nbackground_mCherry: " + str(background_mCherry))
ax1.set_title(titulo_slice, loc='left')

plt.show()
plt.close(fig) 
file_name_slice_image = (output_folder_images_dataset + "/" + str(dataset_date) + "-" + 
                    str(genotype_number) + "-slice_" + str(analysis_slice) + ".png")
print("Output file:", file_name_slice_image, "\n")
fig.savefig(file_name_slice_image, bbox_inches='tight')

###Generate images of individual cells
###

iteration = 0
for row in selected_cells_stats:

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={'width_ratios': [1, 1, 1]})
 
    
    file_name_image = (output_folder_images_dataset + "/fluo_" + str(dataset_date) + "-" + str(genotype_number) 
                       + "-slice_" + str(analysis_slice) + "-cell_" + str(int(row[0])) + ".png")
    
    print(file_name_image)
    #+-450 for SoRa
    #+-200 for Confocal
    plot_y_low = selected_coordinates[iteration][0]
    plot_y_up = selected_coordinates[iteration][1]
    plot_x_low = selected_coordinates[iteration][2]
    plot_x_up = selected_coordinates[iteration][3]
    
    plot_x_low = 0 if plot_x_low < 0 else plot_x_low
    plot_y_low = 0 if plot_y_low < 0 else plot_y_low
    
    #Contrast enhancing for spy650 channel
    ax1.imshow(image_chNuclei_selected, origin='upper', cmap = 'gray', vmin = vmin_nuc, vmax = vmax_nuc)
    ax1.set_xlim(plot_x_low, plot_x_up) #zoom to mask currently analyzed
    ax1.set_ylim(plot_y_up, plot_y_low)
    
    #Add scale bar on ax1 image
    #76um x 76um field view size (1024 x 1024 pixels). Pixel size is 0.073 um (73 nm).
    #Thus, 137 pixels is equivalent to 10 000 nm 
    scale_color='white'
    scale_units='μm'
    scale_length = 10  # in data coordinates (micrometers)
    pixel2micrometer_ratio = 1 / 0.073 #ratio of 1 pixel to distance units
    scale_location = (plot_x_low + 50, plot_y_up - 50)  # starting point of the scale bar
    
    rect = Rectangle(scale_location, scale_length * pixel2micrometer_ratio, 16, edgecolor='black', facecolor=scale_color)
    ax1.add_patch(rect)
    ax1.text(scale_location[0] + scale_length * pixel2micrometer_ratio / 2, scale_location[1] + 0.04,
            f'{scale_length} {scale_units}', color=scale_color, ha='center', va='bottom', fontsize = 20)
    ####
    ####
    #plot mEGFP
    vmin = np.percentile(image_chmEGFP_selected, 20) #set contrast
    vmax = np.percentile(image_chmEGFP_selected, 99.8) #set contrast
    im = ax2.imshow(image_chmEGFP_selected, origin='upper', vmin = vmin, vmax = vmax)
    ax2.set_xlim(plot_x_low, plot_x_up) #zoom to mask currently analyzed
    ax2.set_ylim(plot_y_up, plot_y_low)
    cbar = plt.colorbar(im, ax=ax2, fraction=0.04, pad=0.04) # Add a colorbar to the first subplot
    ax2.set_title("mEGFP")
    
    #plot mCherry
    vmin = np.percentile(image_chmCherry_selected, 20) #set contrast
    vmax = np.percentile(image_chmCherry_selected, 99.8) #set contrast
    im = ax3.imshow(image_chmCherry_selected, origin='upper', vmin = vmin, vmax = vmax)
    ax3.set_xlim(plot_x_low, plot_x_up) #zoom to mask currently analyzed
    ax3.set_ylim(plot_y_up, plot_y_low)
    cbar = plt.colorbar(im, ax=ax3, fraction=0.04, pad=0.04) # Add a colorbar to the first subplot
    ax3.set_title("mCherry")

    # Adjust spacing between subplots
    plt.tight_layout()
    titulo = ("date: " +str(dataset_date) + "\ndataset: " + str(genotype_number) + "\nslice: " + str(analysis_slice) + 
                       "\ncell: " + str(int(row[0])))
    ax1.set_title(titulo, loc='left')

    # Save the figure as PNG with specified DPI
    print("Output file:", file_name_image)
    fig.savefig(file_name_image, bbox_inches='tight')
    plt.show()
    plt.close(fig) 
    
    iteration = iteration + 1


In [ ]:
## Generate a file for condensates that includes masks 
###Generate a file for masks

# selected_cells_stats = np.append(selected_cells_stats, np.array([[mask_number, 
#                center_cell_x, center_cell_y, raw_mEGFP_cell_expression, raw_mCherry_cell_expression, 
#                cell_area, background_mEGFP, background_mCherry]]), axis=0)

rows2write_cells = []

headers_cells = ("date" + "\t" + "dataset" + "\t" + "z-slice" + "\t" + "cell_Id" +
            "\t" + "x" + "\t" + "y" + "\t" + "raw_exp_mEGFP" + "\t" + "raw_exp_mCherry" +
            "\t" +"cell_area" + "\t" + "background_mEGFP" + "\t" + "background_mCherry\n")
rows2write_cells.append(headers_cells)

for row in selected_cells_stats:
        
    data_row = (str(dataset_date) + "\t" + str(genotype_number) + "\t" + str(analysis_slice) + 
            "\t" + str(row[0]) + "\t" + str(row[1]) + "\t" + str(row[2]) + 
            "\t" + str(row[3]) + "\t" + str(row[4]) + "\t" + str(row[5]) + 
            "\t" + str(row[6]) + "\t" + str(row[7]) + "\n")
    
    rows2write_cells.append(data_row)
    
#print(rows2write_cells)
#Create folder and files for analysis
# Specify the folder path
if not os.path.exists(output_folder): # Check if the folder exists
    #Create the folder
    os.makedirs(output_folder)
    print("Folder created successfully!")
else:
    print("Folder already exists.")

try:
    with open(filename_cells, 'w') as f:
        for row in rows2write_cells:
            f.write(row)
            
except IOError:
    print("Could not open file {filename_cells} for writing") #Handle the case where the file cannot be opened or written to

print("File:\n", filename_cells, "\ngenerated")


In [ ]:
###### Generate a file for the parameters used in the current analysis

##Get file name of the script (for Jupyter)
##

##Obtain date
from datetime import datetime
current_date = datetime.now()
formatted_date = current_date.strftime("%Y-%m-%d")

code_date_parameters = 'NSPARC-' + formatted_date + "\n"
date_parameters = "date: " + str(dataset_date) + "\n"
dataset_parameters = "dataset: " + str(genotype_number) + "\n"
z_slice_parameters = "z-slice: " + str(analysis_slice) + "\n"
model_parameters = "cell segmentation model: cellpose-" + str(model_type) + "\n"
Comments_parameters = "Comments: " + Comments + "\n"

print(code_date_parameters)
print(dataset_parameters)
print(z_slice_parameters)
print(model_parameters)
print(Comments_parameters)
# rows2write_parameters = []


if not os.path.exists(output_folder): # Check if the folder exists
    #Create the folder
    os.makedirs(output_folder)
    print("Folder created successfully!")
else:
    print("Folder already exists.")

try:
    with open(filename_parameters, 'w') as f:
        f.write(code_date_parameters)
        f.write(dataset_parameters)
        f.write(z_slice_parameters)
        f.write(model_parameters)
        f.write(Comments_parameters)
            
except IOError:
    print("Could not open file {filename_parameters} for writing") #Handle the case where the file cannot be opened or written to

print("File:\n", filename_parameters, "\ngenerated")
